# UMAP of LeWM embeddings

Encode every state of one dataset with several trained LeWMs, reduce the 192-d embeddings to 2-D with UMAP and colour the points by level, success, progress within the episode and action.

UMAP distorts distances and cluster sizes: use the plots for intuition, not as a measurement.

Kernel: the project's `.venv` (Python 3.12).

In [1]:
import io
import json
import os

os.environ.setdefault("STABLEWM_HOME", "/home/lukas/TU_Dresden/master_thesis/data/stable_worldmodel")  # the notebook kernel may not see the shell's export

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import umap
from hydra.utils import instantiate
from omegaconf import OmegaConf
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

from master_thesis.evaluation.arc_dataset_metric_visualization import load_columns  # same column loading as the dataset metrics
from master_thesis.paths import evaluation_dir, model_dir
from master_thesis.training.lewm import ArcGridToPixels  # same preprocessing as in training

01:40:28 | INFO  | __init__.py | JAX version 0.6.2 available.
01:40:33 | INFO  | atomic_chec~| [atomic_save] installed crash-safe checkpoint plugin (write to sibling .tmp + fsync + atomic rename)


In [2]:
GAME = "ls20"
DATASET = "human_l1-7"  # states to embed (none of the models below was trained on human data)
MODELS = [  # folder names in models/lewm
    "ls20_goose-l1-7_10ep_0914-0027",
    "ls20_ppo-l1-7_ft20ep_0914-1423",
    "ls20_goose-ppo-l1_ft130ep_0914-2034",
]
SEED = 0  # UMAP seed -> the same layout every time
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
grids, actions, levels, successes, episodes = load_columns(GAME, DATASET)  # one row per state

progress = np.zeros(len(grids))  # 0 = first state of its episode, 1 = last state
episode_of_row = np.zeros(len(grids), dtype=int)  # which episode each row belongs to
for index, (start, length) in enumerate(episodes):
    progress[start:start + length] = np.arange(length) / max(length - 1, 1)
    episode_of_row[start:start + length] = index

print(f"{len(grids)} states, {len(episodes)} episodes, levels {sorted(set(levels.tolist()))}")

01:40:33 | WARN  | __init__.py | Skipping forkserver preload of torchcodec (Could not load this library: /home/lukas/TU_Dresden/master_thesis/thesis_repo/.venv/lib/python3.12/site-packages/torchcodec/libtorchcodec_image.so)
7642 states, 117 episodes, levels [1, 2, 3, 4, 5, 6, 7]


In [4]:
def load_lewm(run_name):
    """Rebuild a trained LeWM from models/lewm/<run_name> (as in training/ppo.py)."""
    run = model_dir("lewm", run_name)
    model = instantiate(json.loads((run / "model_config.json").read_text()))  # architecture
    model.load_state_dict(torch.load(run / "weights.pt", map_location="cpu", weights_only=True))  # trained weights
    img_size = int(OmegaConf.load(run / "train_config.yaml").img_size)  # image size used in training
    return model.to(DEVICE).eval(), img_size


@torch.no_grad()
def encode_states(model, img_size, batch_size=256):
    """Embed every grid of the dataset: [states, 4096] -> [states, 192]."""
    to_pixels = ArcGridToPixels(img_size)
    embeddings = []
    for start in range(0, len(grids), batch_size):  # batches keep GPU memory small
        batch = torch.as_tensor(grids[start:start + batch_size], device=DEVICE)  # [B, 4096]
        pixels = to_pixels(batch)  # [B, 3, 224, 224]
        emb = model.encode({"pixels": pixels.unsqueeze(1)})["emb"][:, 0]  # encode expects [B, time, ...]; take time step 0 -> [B, 192]
        embeddings.append(emb.float().cpu().numpy())
    return np.concatenate(embeddings)

In [5]:
embeddings = {}  # model name -> [states, 192]
for name in MODELS:
    model, img_size = load_lewm(name)
    embeddings[name] = encode_states(model, img_size)
    del model  # free GPU memory before the next model
    torch.cuda.empty_cache()
    print(name, embeddings[name].shape)

01:40:36 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}
ls20_goose-l1-7_10ep_0914-0027 (7642, 192)
01:40:43 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}
ls20_ppo-l1-7_ft20ep_0914-1423 (7642, 192)
01:40:49 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}
ls20_goose-ppo-l1_ft130ep_0914-2034 (7642, 192)


In [6]:
layouts = {}  # model name -> [states, 2]
for name in MODELS:
    reducer = umap.UMAP(n_neighbors=50, min_dist=0.1, init="pca", random_state=SEED)  # 50 neighbours connect the many near-identical states; PCA start instead of the spectral start (which fails here)
    layouts[name] = reducer.fit_transform(embeddings[name])
    print(name, "done")

/home/lukas/TU_Dresden/master_thesis/thesis_repo/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


ls20_goose-l1-7_10ep_0914-0027 done


/home/lukas/TU_Dresden/master_thesis/thesis_repo/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


ls20_ppo-l1-7_ft20ep_0914-1423 done


/home/lukas/TU_Dresden/master_thesis/thesis_repo/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


ls20_goose-ppo-l1_ft130ep_0914-2034 done


In [7]:
colourings = [  # (title, values, colour map, discrete?)
    ("level", levels, "tab10", True),
    ("episode completed its level", successes.astype(int), "coolwarm", True),
    ("progress within episode", progress, "viridis", False),
    ("action taken (0 = last state)", actions, "tab10", True),
]

figure, axes = plt.subplots(len(MODELS), len(colourings), figsize=(5 * len(colourings), 4.5 * len(MODELS)), squeeze=False)
for row, name in enumerate(MODELS):
    for column, (title, values, cmap, discrete) in enumerate(colourings):
        axis = axes[row, column]
        points = axis.scatter(layouts[name][:, 0], layouts[name][:, 1], c=values, cmap=cmap, s=2)
        axis.set_xticks([])
        axis.set_yticks([])
        axis.set_title(f"{title}" if row else f"{title}\n", fontsize=10)
        if column == 0:
            axis.set_ylabel(name, fontsize=9)
        if discrete:
            axis.legend(*points.legend_elements(), fontsize=7, markerscale=0.7, loc="best")  # one entry per value
        else:
            figure.colorbar(points, ax=axis, fraction=0.046)
figure.suptitle(f"UMAP of LeWM embeddings, states of {DATASET}")
figure.tight_layout()

output = evaluation_dir(GAME) / f"{DATASET}_umap_lewm.png"  # file name starts with the dataset
output.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(output, dpi=120)
print("Saved", output)

Saved /home/lukas/TU_Dresden/master_thesis/data/stable_worldmodel/evaluation/ls20/human_l1-7_umap_lewm.png


## One episode as a path

Draw the states of a single episode in order on top of all states (grey). A smooth path means consecutive states stay close in embedding space.

In [8]:
EPISODE = 52  # episode index (same numbers as in the GIF file names)

start, length = episodes[EPISODE]
figure, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4.5), squeeze=False)
for axis, name in zip(axes[0], MODELS):
    layout = layouts[name]
    axis.scatter(layout[:, 0], layout[:, 1], c="lightgrey", s=1)  # all states
    path = layout[start:start + length]
    axis.plot(path[:, 0], path[:, 1], color="black", linewidth=0.5)  # order of the states
    points = axis.scatter(path[:, 0], path[:, 1], c=np.arange(length), cmap="plasma", s=8)  # colour = step
    axis.set_title(name, fontsize=9)
    axis.set_xticks([])
    axis.set_yticks([])
figure.colorbar(points, ax=axes[0].tolist(), fraction=0.02, label="step")
figure.suptitle(f"{DATASET}: episode {EPISODE} (level {levels[start]}, {'completed' if successes[start] else 'not completed'})")

Text(0.5, 0.98, 'human_l1-7: episode 52 (level 2, completed)')

## PCA (linear, no settings)

Sanity check for UMAP: if levels also separate here, the effect is not a UMAP artifact. SIGReg pushes the embeddings towards an isotropic Gaussian, so two PCA axes may explain only a small part of the variance.

In [9]:
figure, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4.5), squeeze=False)
for axis, name in zip(axes[0], MODELS):
    pca = PCA(n_components=2).fit(embeddings[name])  # the two directions with the most variance
    points = pca.transform(embeddings[name])  # [states, 2]
    scatter = axis.scatter(points[:, 0], points[:, 1], c=levels, cmap="tab10", s=2)
    axis.set_title(f"{name}\n{100 * pca.explained_variance_ratio_.sum():.0f}% of the variance", fontsize=9)  # how much of the 192-d spread the 2 axes show
    axis.set_xticks([])
    axis.set_yticks([])
    axis.legend(*scatter.legend_elements(), fontsize=7, markerscale=0.7, title="level")
figure.suptitle(f"PCA of LeWM embeddings, states of {DATASET}")
figure.tight_layout()
figure.savefig(evaluation_dir(GAME) / f"{DATASET}_pca_lewm.png", dpi=120)

## Nearest neighbour check (in the original 192-d space)

For every state, take its closest state **from another episode** (neighbours from the same episode would trivially share the level) and check whether it plays the same level. A high share means the model keeps the levels apart.

In [10]:
NEIGHBOURS = 30  # how many neighbours to search for the first one from another episode

table = {}  # model name -> share of states whose neighbour has the same level, per level
for name in MODELS:
    _, index = NearestNeighbors(n_neighbors=NEIGHBOURS).fit(embeddings[name]).kneighbors(embeddings[name])  # [states, NEIGHBOURS], sorted by distance
    other_episode = episode_of_row[index] != episode_of_row[:, None]  # True where the neighbour is from another episode
    found = other_episode.any(axis=1)  # states with at least one such neighbour among the 30
    neighbour = index[np.arange(len(index)), other_episode.argmax(axis=1)]  # the closest neighbour from another episode
    same_level = levels[neighbour] == levels  # does it play the same level?
    table[name] = {f"level {level}": same_level[found & (levels == level)].mean() for level in sorted(set(levels.tolist()))}
    table[name]["all"] = same_level[found].mean()

pd.DataFrame(table).round(2)  # rows = levels, columns = models; 1.0 = neighbours always from the same level

,ls20_goose-l1-7_10ep_0914-0027,ls20_ppo-l1-7_ft20ep_0914-1423,ls20_goose-ppo-l1_ft130ep_0914-2034
level 1,0.90,0.90,0.69
level 2,1.00,1.00,0.93
level 3,0.98,0.98,0.84
level 4,0.97,0.92,0.88
level 5,0.98,0.97,0.93
level 6,0.99,0.99,0.89
level 7,1.00,1.00,0.92
all,0.98,0.98,0.89


## Interactive 3-D UMAP (plotly)

Rotate with the mouse; hovering shows episode, step, action and whether the episode completed its level. The episode number matches the GIF file names (`arc_dataset_metric_visualization --gifs`). Also saved as an HTML file in `evaluation/<game>/`.

In [11]:
PLOTLY_MODEL = MODELS[2]  # which model to show in 3-D

layout_3d = umap.UMAP(n_components=3, n_neighbors=50, min_dist=0.1, init="pca", random_state=SEED).fit_transform(embeddings[PLOTLY_MODEL])  # same settings as above, but 3 axes
episode_starts = np.array([start for start, _ in episodes])  # first row of every episode
points_3d = pd.DataFrame({
    "x": layout_3d[:, 0], "y": layout_3d[:, 1], "z": layout_3d[:, 2],
    "level": levels.astype(str),  # text -> one colour per level
    "episode": episode_of_row,  # same number as in the GIF file names
    "step": np.arange(len(grids)) - episode_starts[episode_of_row],  # position within the episode
    "action": actions,
    "completed": successes,
})

figure_3d = px.scatter_3d(points_3d, x="x", y="y", z="z", color="level", hover_data=["episode", "step", "action", "completed"], category_orders={"level": sorted(points_3d["level"].unique(), key=int)}, title=f"3-D UMAP, {PLOTLY_MODEL}, states of {DATASET}")
figure_3d.update_traces(marker_size=2)  # small points
figure_3d.write_html(evaluation_dir(GAME) / f"{DATASET}_umap3d_{PLOTLY_MODEL}.html")  # interactive file that opens in any browser
figure_3d.show()

/home/lukas/TU_Dresden/master_thesis/thesis_repo/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
PLOTLY_MODEL = MODELS[2]  # which model to show in 3-D

layout_3d = umap.UMAP(n_components=3, n_neighbors=50, min_dist=0.1, init="pca", random_state=SEED).fit_transform(embeddings[PLOTLY_MODEL])  # same settings as above, but 3 axes
episode_starts = np.array([start for start, _ in episodes])  # first row of every episode
points_3d = pd.DataFrame({
    "x": layout_3d[:, 0], "y": layout_3d[:, 1], "z": layout_3d[:, 2],
    "level": levels.astype(str),  # text -> one colour per level
    "episode": episode_of_row,  # same number as in the GIF file names
    "step": np.arange(len(grids)) - episode_starts[episode_of_row],  # position within the episode
    "action": actions,
    "completed": successes,
})

figure_3d = px.scatter_3d(points_3d, x="x", y="y", z="z", color="level", hover_data=["episode", "step", "action", "completed"], category_orders={"level": sorted(points_3d["level"].unique(), key=int)}, title=f"3-D UMAP, {PLOTLY_MODEL}, states of {DATASET}")
figure_3d.update_traces(marker_size=2)  # small points
figure_3d.write_html(evaluation_dir(GAME) / f"{DATASET}_umap3d_{PLOTLY_MODEL}.html")  # interactive file that opens in any browser
figure_3d.show()

/home/lukas/TU_Dresden/master_thesis/thesis_repo/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 3-D UMAP with the hovered state's grid (live widget)

Uses `figure_3d` from the 3-D cell that ran last. Hover over a point to see that state's grid on the right. Works only while the notebook kernel is running (not in the saved HTML).

In [13]:
palette = (ArcGridToPixels(64).palette.numpy() * 255).round().astype(np.uint8)  # [16, 3] colour index -> RGB


def grid_png(row, scale=4):
    """PNG bytes of one state, enlarged `scale` times with sharp pixels."""
    rgb = palette[grids[row].reshape(64, 64)]  # [64, 64, 3]
    image = Image.fromarray(rgb).resize((64 * scale, 64 * scale), Image.NEAREST)
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return buffer.getvalue()


state_image = widgets.Image(format="png", width=256, height=256)  # shows the hovered state
state_caption = widgets.HTML("hover over a point")  # text below the image


def show_state(trace, points, state):
    """Called by plotly when the mouse is over a point: show that state's grid."""
    if not points.point_inds:  # mouse is not over a point of this trace
        return
    episode, step = trace.customdata[points.point_inds[0]][:2]  # hover_data order: episode, step, action, completed
    row = episode_starts[int(episode)] + int(step)  # row of this state in grids
    state_image.value = grid_png(row)
    state_caption.value = f"episode {int(episode)} · step {int(step)} · level {levels[row]} · action {actions[row]} · {'completed' if successes[row] else 'not completed'}"


widget_figure = go.FigureWidget(figure_3d)  # interactive copy of the last 3-D figure above
widget_figure.update_layout(width=750, height=650)
for trace in widget_figure.data:  # px makes one trace per level colour
    trace.on_hover(show_state)

widgets.HBox([widget_figure, widgets.VBox([state_image, state_caption])])  # plot left, state right

    'data': [{'customdata': array([[0, 0, 4, True],
                            …